<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/data/data.zip
!unzip data.zip


--2025-05-19 19:37:10--  https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/data/data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 207937 (203K) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>] 203.06K  --.-KB/s    in 0.03s   

2025-05-19 19:37:10 (5.81 MB/s) - ‘data.zip’ saved [207937/207937]

Archive:  data.zip
  inflating: bh-1.json               
  inflating: bh-2.json               
  inflating: bh-3.json               
  inflating: bh-4.json               
  inflating: bh-5-6016-5981.json     
  inflating: bh-6-all_tier_0.json    
  inflating: bh-7.json               
  inflating: bh-10.json              
  inflating: bh-all_tier_0.json      
  inflating: bh_all_tier_0-formatted.json  
  inflat

In [3]:
!mkdir/content/data
!unzip data.zip -d /content/data


/bin/bash: line 1: mkdir/content/data: No such file or directory
Archive:  data.zip
  inflating: /content/data/bh-1.json  
  inflating: /content/data/bh-2.json  
  inflating: /content/data/bh-3.json  
  inflating: /content/data/bh-4.json  
  inflating: /content/data/bh-5-6016-5981.json  
  inflating: /content/data/bh-6-all_tier_0.json  
  inflating: /content/data/bh-7.json  
  inflating: /content/data/bh-10.json  
  inflating: /content/data/bh-all_tier_0.json  
  inflating: /content/data/bh_all_tier_0-formatted.json  
  inflating: /content/data/bh-graph.json  
  inflating: /content/data/DORTHY_ROWE-ROSEANN_FISCHER.json  
  inflating: /content/data/highlighted-path.png  
  inflating: /content/data/tier0.json  


In [19]:
# prompt: import bh.json and load into a pandas dataframe df

!pip install pandas

import pandas as pd
import json

# Assuming 'bh.json' is in the current working directory
with open('data/bh-unconstrained.json', 'r') as f:
    data_all = json.load(f)

df_all = pd.DataFrame(data_all)

# Display the first few rows of the DataFrame to verify
print(df_all.head())

In [20]:
# prompt: from df_all['data'].keys() extract the 'nodes' into a seperate dataframe called nodes_all. i want the vlues as rows and the column headers as ID, name, kind, group, SID, OID, istier0, isOwned, lastSeen.

import pandas as pd
pd.set_option("display.max_colwidth", None)
nodes_all = pd.DataFrame(df_all['data']['nodes']).T
nodes_all.columns = ['label', 'kind', 'OID', 'istier0', 'isOwned', 'lastSeen']
#print(nodes_all)


In [ ]:
# prompt: convert the keys for nodes_all to an integer and then create a seperate index on nodes_all starting at 0

nodes_all.index = nodes_all.index.astype(int)
nodes_all = nodes_all.reset_index().rename(columns={'index': 'node_id'})
nodes_all

In [22]:
# prompt: create a seperate dataframe from df_all using the edges key

import pandas as pd
edges_all = pd.DataFrame(df_all['data']['edges'])
print(edges_all)

     source target       label        kind                        lastSeen
0         5   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
1         6   6264    MemberOf    MemberOf  2025-05-13T13:28:36.181502638Z
2      6264   5186    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
3      5186   5483  GenericAll  GenericAll  2025-05-13T13:28:36.832245798Z
4      5483   5179    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
...     ...    ...         ...         ...                             ...
3953   5178   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
3954   6689   6259    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
3955   6259   5168  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
3956   5168   5178  GenericAll  GenericAll  2025-05-13T13:28:44.995877259Z
3957   5178   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z

[3958 rows x 5 columns]


In [23]:
# prompt: add a weight column to the edges_all with the following criteria:

# Apply the weights based on the 'label' column
def assign_weight(label):
    if label == "GenericAll":
        return 9
    elif label == "Owns":
        return 10
    elif label == "GenericWrite":
        return 2
    elif label == "AllExtendedRights":
        return 6
    elif label == "CanRDP":
        return 2
    elif label == "Contains":
        return 2
    elif label == "DCSync":
        return 8
    elif label == "WriteDacl":
        return 5
    elif label == "WriteOwner":
        return 7
    elif label == "AddKeyCredentialLink":
        return 6
    elif label == "AdminTo":
        return 8
    elif label == "MemberOf":
        return 1
    elif label == "CanPSRemote":
        return 2
    elif label == "ExecuteDCOM":
        return 2
    elif label == "GPLink":
        return 3
    elif label == "tier0":
        return 0

    else:
        return 1 # Default weight for other labels

edges_all['weight'] = edges_all['label'].apply(assign_weight)

# Display the updated edges_all DataFrame with the 'weight' column
print("\nEdges_all DataFrame with weight column:")
edges_all


Edges_all DataFrame with weight column:


,source,target,label,kind,lastSeen,weight
0,5,5173,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
1,6,6264,MemberOf,MemberOf,2025-05-13T13:28:36.181502638Z,1
2,6264,5186,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
3,5186,5483,GenericAll,GenericAll,2025-05-13T13:28:36.832245798Z,9
4,5483,5179,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
...,...,...,...,...,...,...
3953,5178,5173,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
3954,6689,6259,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
3955,6259,5168,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
3956,5168,5178,GenericAll,GenericAll,2025-05-13T13:28:44.995877259Z,9


In [24]:
# prompt: convert edges_all['source') to int

edges_all['source'] = edges_all['source'].astype(int)
edges_all['target'] = edges_all['target'].astype(int)

print("\nConverted 'source' column to int:")
print(edges_all['source'].head())
print("\nConverted 'target' column to int:")
print(edges_all['target'].head())

# Verify the data types
print("\nData types after conversion:")
edges_all.dtypes


Converted 'source' column to int:
0       5
1       6
2    6264
3    5186
4    5483
Name: source, dtype: int64

Converted 'target' column to int:
0    5173
1    6264
2    5186
3    5483
4    5179
Name: target, dtype: int64

Data types after conversion:


,0
source,int64
target,int64
label,object
kind,object
lastSeen,object
weight,int64


In [25]:
# prompt: how would i use the PageRank algorythm with nodes_all and edges_all data

!pip install networkx
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes to the graph
# You can add nodes with attributes from your nodes_all DataFrame if needed
for index, row in nodes_all.iterrows():
    G.add_node(row['node_id'], label=row['label'], kind=row['kind'], OID=row['OID'], istier0=row['istier0'], isOwned=row['isOwned'], lastSeen=row['lastSeen'])

# Add edges to the graph with weights
for index, row in edges_all.iterrows():
    G.add_edge(row['source'], row['target'], weight=row['weight'])

# Calculate PageRank
# alpha is the damping parameter, commonly set to 0.85
pagerank_scores = nx.pagerank(G, alpha=0.85, weight='weight')

# You can now access the PageRank score for each node
# For example, to see the scores:
print("\nPageRank scores:")
for node, score in pagerank_scores.items():
    print(f"Node {node}: {score:.4f}")

# To add the PageRank scores back to your nodes_all DataFrame:
nodes_all['pagerank'] = nodes_all['node_id'].map(pagerank_scores)

print("\nNodes_all DataFrame with PageRank scores:")
print(nodes_all[['node_id', 'label', 'pagerank']].head(12))

# You can then sort the nodes by PageRank to find the most important nodes
print("\nTop 10 nodes by PageRank:")
print(nodes_all.sort_values(by='pagerank', ascending=False).head(12))


PageRank scores:
Node 5: 0.0028
Node 6: 0.0003
Node 5075: 0.0003
Node 5076: 0.0003
Node 5077: 0.0003
Node 5078: 0.0003
Node 5079: 0.0003
Node 5080: 0.0003
Node 5081: 0.0003
Node 5082: 0.0003
Node 5083: 0.0003
Node 5084: 0.0003
Node 5085: 0.0007
Node 5086: 0.0003
Node 5087: 0.0007
Node 5088: 0.0003
Node 5089: 0.0003
Node 5090: 0.0003
Node 5091: 0.0003
Node 5092: 0.0005
Node 5093: 0.0003
Node 5094: 0.0005
Node 5095: 0.0005
Node 5096: 0.0003
Node 5097: 0.0003
Node 5098: 0.0003
Node 5099: 0.0003
Node 5100: 0.0005
Node 5101: 0.0003
Node 5102: 0.0003
Node 5103: 0.0003
Node 5104: 0.0003
Node 5105: 0.0003
Node 5106: 0.0003
Node 5107: 0.0003
Node 5108: 0.0027
Node 5109: 0.0003
Node 5110: 0.0003
Node 5111: 0.0003
Node 5112: 0.0003
Node 5113: 0.0003
Node 5114: 0.0003
Node 5115: 0.0003
Node 5116: 0.0005
Node 5117: 0.0003
Node 5118: 0.0003
Node 5119: 0.0003
Node 5120: 0.0003
Node 5121: 0.0003
Node 5122: 0.0003
Node 5123: 0.0003
Node 5124: 0.0003
Node 5125: 0.0003
Node 5126: 0.0003
Node 5127: 0.000

In [27]:
# prompt: can you sort the pagerank score from hightest to lowest

# Sort the DataFrame by pagerank in descending order
sorted_pagerank = nodes_all.sort_values(by='pagerank', ascending=False)

# Print the sorted DataFrame
print("\nNodes sorted by PageRank (highest to lowest):")
print(sorted_pagerank[['node_id', 'label', 'pagerank', 'kind']])


Nodes sorted by PageRank (highest to lowest):
     node_id                               label  pagerank      kind
100     5173                  DCTEST.MYLAB.LOCAL  0.151007  Computer
105     5178  MA-VACACIONE-DISTLIST1@MYLAB.LOCAL  0.045157     Group
104     5177        DE-DAS-DISTLIST1@MYLAB.LOCAL  0.039653     Group
106     5179        RO-ARJ-DISTLIST1@MYLAB.LOCAL  0.038455     Group
775     6264        DOMAIN COMPUTERS@MYLAB.LOCAL  0.022469     Group
..       ...                                 ...       ...       ...
458     5853             GAIL_ALSTON@MYLAB.LOCAL  0.000252      User
460     5857             LUELLA_SOTO@MYLAB.LOCAL  0.000252      User
461     5862           FLORA_CLAYTON@MYLAB.LOCAL  0.000252      User
463     5866          KARLA_RICHARDS@MYLAB.LOCAL  0.000252      User
445     5825             NINA_HARRIS@MYLAB.LOCAL  0.000252      User

[1106 rows x 4 columns]
